# Phase 5: Sustainability Tracking (energy and CO₂)

This is the part most papers skip: how much **energy** does training each model use, and how much **CO₂** does that correspond to?

We measure it with **CodeCarbon**, an open-source tool that runs alongside our code while a model trains. It estimates:
- **energy (kWh)** used by the CPU and the memory during training
- **CO₂ emissions (kg)** = energy × the carbon intensity of the electricity grid where the computer is. CodeCarbon detects the location (cloud region, or country) and we take that location's intensity from CodeCarbon's own data tables.

*Why we compute CO₂ from energy ourselves:* in testing, CodeCarbon's built-in CO₂ total didn't divide consistently by its energy total for short runs. The implied grid intensity jumped between 0.8 and 1.7 kg/kWh for the same location within minutes, because the two totals are summed at slightly different moments. Energy is the quantity actually measured, so we use CO₂ = energy × (one fixed intensity for the location). Every model then gets exactly the same grid factor. CodeCarbon's raw figure is kept in `phase5_energy_all_raw.csv` (`co2_codecarbon_raw_kg`) for transparency.

**Which models we measure (agreed by the team on 4 Oct 2026):**
- **The tuned version of every model** (settings from Phase 4). These are the models a provider would actually deploy, and their numbers go into the TOPSIS ranking in Phase 7.
- **The default version** (Phase 3 settings) as a reference, so the paper can show how tuning changed the energy cost.
- **The energy spent on tuning itself**, estimated separately (see Section 4).

**An honest caveat about cloud measurement:** Colab's virtual machines don't expose the processor's real power sensors. CodeCarbon therefore *estimates* CPU power from the processor's rated power (TDP), and memory power from the amount of RAM. Energy is then roughly *power × time*, so on Colab energy and CO₂ follow training time closely. The comparison between models is still fair, because all of them run on the same machine in the same session, but the paper has to state this in its Limitations section. The exact method CodeCarbon used is logged below.

**Very fast models:** Linear Regression trains in about 0.3 s, too quick to measure on its own. Any model that trains in under 30 seconds is trained repeatedly until about 30 seconds have passed, and the total energy is divided by the number of repeats. All numbers below are **per single training run**.

**Run time on free Colab:** about 30–40 minutes (Random Forest twice, the LSTM twice, everything else about 30 s each). **If Colab disconnects, run everything again.** Each measurement is saved as soon as it finishes, and finished ones are skipped.

## 0. Setup

In [3]:
# CodeCarbon isn't preinstalled on Colab
import importlib.util, subprocess, sys
if importlib.util.find_spec('codecarbon') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'codecarbon'], check=True)

In [4]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [5]:
import gc, json, math, os, logging, warnings
import numpy as np
import pandas as pd
import torch
import codecarbon
from codecarbon import EmissionsTracker
import model_definitions as M
importlib.reload(M)

logging.getLogger('codecarbon').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', module='codecarbon')

FORCE_RERUN = False
MIN_MEASURE_SEC = 30      # short trainings are repeated until about this long, then divided

log = U.get_logger('phase05_sustainability_tracking')
log.info(f'codecarbon {codecarbon.__version__} | CPU cores: {os.cpu_count()} | FORCE_RERUN = {FORCE_RERUN}')

RUN STARTED 2026-10-04 06:24:20 | phase05_sustainability_tracking
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project
codecarbon 3.3.1 | CPU cores: 2 | FORCE_RERUN = False


## 1. Load data, default settings and tuned settings

In [6]:
train = pd.read_parquet(C.DATA_PROCESSED / 'train.parquet')
features_main = pd.read_parquet(C.DATA_PROCESSED / 'features_main.parquet')
X_train, y_train = train[U.FEATURE_COLUMNS], train[C.TARGET].to_numpy()

phase3 = json.load(open(C.MODELS_DIR / 'phase3_results.json'))
phase4 = json.load(open(C.MODELS_DIR / 'phase4_results.json'))

# Settings for each (model, version). Default = library defaults; tuned = Phase 4 winner.
configs = {}
for name in M.TABULAR_MODELS + ['LSTM']:
    tuned_params = phase4[name]['best_params'] if name in phase4 else {}
    configs[(name, 'default')] = {'params': {}, 'expected_sec': phase3[name]['train_time_sec']}
    configs[(name, 'tuned')] = {'params': tuned_params,
                                'expected_sec': phase4.get(name, phase3[name])['train_time_sec']}
for (name, version), cfg in configs.items():
    log.info(f'{name:<17} {version:<7} params={cfg["params"]}  (expected train time ~{cfg["expected_sec"]:.1f}s)')

RESULTS_FILE = C.MODELS_DIR / 'phase5_results.json'
results = {} if FORCE_RERUN or not RESULTS_FILE.exists() else json.load(open(RESULTS_FILE))
if results:
    log.info(f'Already measured in an earlier run (will be skipped): {list(results)}')

Linear Regression default params={}  (expected train time ~0.3s)
Linear Regression tuned   params={}  (expected train time ~0.3s)
Decision Tree     default params={}  (expected train time ~11.5s)
Decision Tree     tuned   params={'max_depth': 5, 'min_samples_split': 10}  (expected train time ~3.0s)
Random Forest     default params={}  (expected train time ~611.5s)
Random Forest     tuned   params={'max_depth': 10, 'n_estimators': 100}  (expected train time ~233.0s)
XGBoost           default params={}  (expected train time ~3.2s)
XGBoost           tuned   params={'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 300}  (expected train time ~5.7s)
LightGBM          default params={}  (expected train time ~2.3s)
LightGBM          tuned   params={'learning_rate': 0.05, 'max_depth': -1, 'n_estimators': 300}  (expected train time ~9.1s)
LSTM              default params={}  (expected train time ~217.5s)
LSTM              tuned   params={'hidden_size': 32, 'seq_len': 12}  (expected train ti

## 2. The measurement function

For each model the steps are: prepare the data (**not** measured) → start CodeCarbon → train the model *n* times → stop CodeCarbon → divide by *n*.

CodeCarbon samples power every second (`measure_power_secs=1`), and its raw readings are appended to `results/logs/phase5_codecarbon_raw.csv` for the reproducibility record.

In [7]:
def grid_carbon_intensity(d):
    """kg CO2 per kWh for the location CodeCarbon detected, from CodeCarbon's own data tables.

    CodeCarbon's built-in emissions total adds up readings at slightly different moments than
    its energy total, so for short runs emissions/energy drifts between runs (we saw 0.8-1.7 in
    testing for one location). We therefore compute CO2 = measured energy x this intensity,
    which is the definition CodeCarbon itself uses, and keep CodeCarbon's raw figure alongside.
    """
    from codecarbon.core.emissions import Emissions
    from codecarbon.core.units import Energy
    from codecarbon.external.geography import CloudMetadata, GeoMetadata
    from codecarbon.input import DataSource
    e = Emissions(DataSource())
    one_kwh = Energy.from_energy(kWh=1)
    if d.cloud_provider and d.cloud_region:
        try:
            return float(e.get_cloud_emissions(one_kwh, CloudMetadata(provider=d.cloud_provider, region=d.cloud_region))), \
                   f'cloud {d.cloud_provider}/{d.cloud_region}'
        except Exception:
            pass
    geo = GeoMetadata(country_iso_code=d.country_iso_code, country_name=d.country_name, region=d.region)
    return float(e.get_country_emissions(one_kwh, geo)), f'country {d.country_iso_code} ({d.region})'

def train_once(name, params, lstm_data=None):
    if name == 'LSTM':
        (Xtr, ytr), (Xva, yva) = lstm_data['train'], lstm_data['val']
        _, history = M.train_lstm(Xtr, ytr, Xva, yva, seed=C.RANDOM_SEED, log_fn=lambda *_: None, **params)
        return len(history)
    model = M.make_model(name, seed=C.RANDOM_SEED, **params)
    model.fit(X_train, y_train)
    del model
    return None

def measure(name, version):
    cfg = configs[(name, version)]
    params = cfg['params']
    repeats = 1 if name == 'LSTM' else int(min(500, max(1, math.ceil(MIN_MEASURE_SEC / max(cfg['expected_sec'], 0.01)))))
    lstm_data = None
    if name == 'LSTM':
        seq_len = params.get('seq_len', M.LSTM_DEFAULTS['seq_len'])
        lstm_data = M.lstm_datasets(features_main, seq_len)
    gc.collect()

    tracker = EmissionsTracker(project_name=f'{name}-{version}', measure_power_secs=1,
                               output_dir=str(C.LOGS_DIR), output_file='phase5_codecarbon_raw.csv',
                               save_to_file=True, log_level='error')
    tracker.start()
    with U.Timer() as t:
        for _ in range(repeats):
            epochs = train_once(name, params, lstm_data)
    tracker.stop()
    d = tracker.final_emissions_data
    intensity, intensity_source = grid_carbon_intensity(d)
    energy_kwh = d.energy_consumed / repeats
    row = {
        'model': name, 'version': version, 'params': params, 'repeats': repeats, 'lstm_epochs': epochs,
        'train_time_sec': t.seconds / repeats,
        'energy_kwh': energy_kwh,
        'cpu_energy_kwh': d.cpu_energy / repeats, 'ram_energy_kwh': d.ram_energy / repeats,
        'co2_emissions_kg': energy_kwh * intensity,
        'co2_codecarbon_raw_kg': d.emissions / repeats,
        'avg_cpu_power_w': d.cpu_power, 'avg_ram_power_w': d.ram_power,
        'carbon_intensity_kg_per_kwh': intensity, 'carbon_intensity_source': intensity_source,
        'country': d.country_name, 'region': d.region, 'cloud_provider': d.cloud_provider,
        'cloud_region': d.cloud_region, 'cpu_model': d.cpu_model, 'cpu_count': d.cpu_count,
        'tracking_mode': d.tracking_mode, 'codecarbon_version': d.codecarbon_version,
    }
    return row

## 3. Measure every model (tuned first, then default)

Tuned versions go first because they're the ones the ranking needs. If time runs short, at least those are saved.

In [8]:
order = [(n, 'tuned') for n in M.TABULAR_MODELS + ['LSTM']] + [(n, 'default') for n in M.TABULAR_MODELS + ['LSTM']]
for name, version in order:
    key = f'{name} | {version}'
    if key in results:
        log.info(f'{key}: already measured, skipping')
        continue
    log.info(f'{key}: measuring ...')
    row = measure(name, version)
    results[key] = row
    json.dump(results, open(RESULTS_FILE, 'w'), indent=2, default=str)
    log.info(f'{key}: {row["train_time_sec"]:.2f}s per training (x{row["repeats"]}) | '
             f'{row["energy_kwh"]:.3e} kWh | {row["co2_emissions_kg"]:.3e} kg CO2 | '
             f'CPU {row["avg_cpu_power_w"]:.1f} W, RAM {row["avg_ram_power_w"]:.1f} W')

Linear Regression | tuned: measuring ...
Linear Regression | tuned: 0.25s per training (x90) | 1.102e-06 kWh | 4.071e-07 kg CO2 | CPU 5.4 W, RAM 10.0 W
Decision Tree | tuned: measuring ...
Decision Tree | tuned: 2.04s per training (x11) | 7.996e-06 kWh | 2.954e-06 kg CO2 | CPU 4.2 W, RAM 10.0 W
Random Forest | tuned: measuring ...
Random Forest | tuned: 194.82s per training (x1) | 9.458e-04 kWh | 3.495e-04 kg CO2 | CPU 7.5 W, RAM 10.0 W
XGBoost | tuned: measuring ...
XGBoost | tuned: 5.92s per training (x6) | 2.247e-05 kWh | 8.303e-06 kg CO2 | CPU 3.6 W, RAM 10.0 W
LightGBM | tuned: measuring ...
LightGBM | tuned: 7.82s per training (x4) | 3.424e-05 kWh | 1.265e-05 kg CO2 | CPU 5.6 W, RAM 10.0 W
LSTM | tuned: measuring ...
LSTM | tuned: 158.91s per training (x1) | 7.593e-04 kWh | 2.805e-04 kg CO2 | CPU 7.2 W, RAM 10.0 W
Linear Regression | default: measuring ...
Linear Regression | default: 0.23s per training (x90) | 1.050e-06 kWh | 3.880e-07 kg CO2 | CPU 6.1 W, RAM 10.0 W
Decision Tre

In [9]:
first = next(iter(results.values()))
log.info('--- MEASUREMENT CONTEXT (for the paper) ---')
for k in ['cpu_model', 'cpu_count', 'tracking_mode', 'country', 'region', 'cloud_provider', 'cloud_region',
          'carbon_intensity_kg_per_kwh', 'carbon_intensity_source', 'avg_cpu_power_w', 'avg_ram_power_w',
          'codecarbon_version']:
    log.info(f'{k:<28}: {first[k]}')

--- MEASUREMENT CONTEXT (for the paper) ---
cpu_model                   : Intel(R) Xeon(R) CPU @ 2.20GHz
cpu_count                   : 2
tracking_mode               : machine
country                     : United States
region                      : iowa
cloud_provider              : 
cloud_region                : 
carbon_intensity_kg_per_kwh : 0.369473
carbon_intensity_source     : country USA (iowa)
avg_cpu_power_w             : 5.449309692747825
avg_ram_power_w             : 10.0
codecarbon_version          : 3.3.1


## 4. Results

**Table 1:** energy and CO₂ of **one training run** of each tuned model. This is what goes into TOPSIS.
**Table 2:** tuned vs default, i.e. what tuning did to the training cost.
**Table 3:** an estimate of the energy spent on the *tuning search itself* (Phase 4). It's the tuning time multiplied by the average power measured while training that same model here, so it's an estimate, not a direct measurement.

CO₂ is also shown in **grams**, because the numbers in kg are tiny.

In [10]:
df = pd.DataFrame(results.values())
tuned = df[df.version == 'tuned'].set_index('model').loc[M.TABULAR_MODELS + ['LSTM']]
energy = pd.DataFrame({
    'Model': tuned.index,
    'Train time (s)': tuned['train_time_sec'].values,
    'Energy (kWh)': tuned['energy_kwh'].values,
    'CO2 (kg)': tuned['co2_emissions_kg'].values,
    'CO2 (g)': tuned['co2_emissions_kg'].values * 1000,
    'Avg power (W)': (tuned['energy_kwh'] * 3.6e6 / tuned['train_time_sec']).values,
    'Repeats measured': tuned['repeats'].values,
})
U.log_df(log, energy, 'TABLE 1: ENERGY AND CO2 PER TRAINING RUN (tuned models)')
U.save_table(energy, 'phase5_energy_tuned', 'Energy and CO2 per training run (tuned models)', float_fmt='{:.3e}')
energy

--- TABLE 1: ENERGY AND CO2 PER TRAINING RUN (tuned models) ---
               Model  Train time (s)  Energy (kWh)      CO2 (kg)   CO2 (g)  Avg power (W)  Repeats measured
0  Linear Regression        0.253325      0.000001  4.070639e-07  0.000407      15.656843                90
1      Decision Tree        2.041978      0.000008  2.954204e-06  0.002954      14.096435                11
2      Random Forest      194.818086      0.000946  3.494571e-04  0.349457      17.477702                 1
3            XGBoost        5.920081      0.000022  8.302546e-06  0.008303      13.664804                 6
4           LightGBM        7.819682      0.000034  1.265218e-05  0.012652      15.765074                 4
5               LSTM      158.907845      0.000759  2.805359e-04  0.280536      17.201364                 1


,Model,Train time (s),Energy (kWh),CO2 (kg),CO2 (g),Avg power (W),Repeats measured
0,Linear Regression,0.253325,0.000001,4.070639e-07,0.000407,15.656843,90
1,Decision Tree,2.041978,0.000008,2.954204e-06,0.002954,14.096435,11
2,Random Forest,194.818086,0.000946,3.494571e-04,0.349457,17.477702,1
3,XGBoost,5.920081,0.000022,8.302546e-06,0.008303,13.664804,6
4,LightGBM,7.819682,0.000034,1.265218e-05,0.012652,15.765074,4
5,LSTM,158.907845,0.000759,2.805359e-04,0.280536,17.201364,1


In [11]:
default = df[df.version == 'default'].set_index('model').loc[M.TABULAR_MODELS + ['LSTM']]
versus = pd.DataFrame({
    'Model': tuned.index,
    'Default energy (kWh)': default['energy_kwh'].values, 'Tuned energy (kWh)': tuned['energy_kwh'].values,
    'Default CO2 (g)': default['co2_emissions_kg'].values * 1000, 'Tuned CO2 (g)': tuned['co2_emissions_kg'].values * 1000,
    'Energy change (%)': (100 * (tuned['energy_kwh'] - default['energy_kwh']) / default['energy_kwh']).values,
})
U.log_df(log, versus, 'TABLE 2: DEFAULT vs TUNED TRAINING ENERGY')
U.save_table(versus, 'phase5_energy_default_vs_tuned', 'Training energy: default vs tuned', float_fmt='{:.3e}')
versus

--- TABLE 2: DEFAULT vs TUNED TRAINING ENERGY ---
               Model  Default energy (kWh)  Tuned energy (kWh)  Default CO2 (g)  Tuned CO2 (g)  Energy change (%)
0  Linear Regression              0.000001            0.000001         0.000388       0.000407           4.922480
1      Decision Tree              0.000049            0.000008         0.018085       0.002954         -83.665241
2      Random Forest              0.002962            0.000946         1.094485       0.349457         -68.071094
3            XGBoost              0.000017            0.000022         0.006414       0.008303          29.444837
4           LightGBM              0.000012            0.000034         0.004261       0.012652         196.896751
5               LSTM              0.001009            0.000759         0.372740       0.280536         -24.736795


,Model,Default energy (kWh),Tuned energy (kWh),Default CO2 (g),Tuned CO2 (g),Energy change (%)
0,Linear Regression,0.000001,0.000001,0.000388,0.000407,4.922480
1,Decision Tree,0.000049,0.000008,0.018085,0.002954,-83.665241
2,Random Forest,0.002962,0.000946,1.094485,0.349457,-68.071094
3,XGBoost,0.000017,0.000022,0.006414,0.008303,29.444837
4,LightGBM,0.000012,0.000034,0.004261,0.012652,196.896751
5,LSTM,0.001009,0.000759,0.372740,0.280536,-24.736795


In [12]:
rows = []
for name in M.TABULAR_MODELS + ['LSTM']:
    tuning_sec = phase4.get(name, {}).get('tuning_time_sec', 0.0)
    power_kw = tuned.loc[name, 'energy_kwh'] / (tuned.loc[name, 'train_time_sec'] / 3600)   # kWh per hour = kW
    kwh = power_kw * tuning_sec / 3600
    rows.append({'Model': name, 'Tuning time (min)': tuning_sec / 60, 'Est. tuning energy (kWh)': kwh,
                 'Est. tuning CO2 (g)': kwh * tuned.loc[name, 'carbon_intensity_kg_per_kwh'] * 1000,
                 'Tuning / one training (x)': tuning_sec / tuned.loc[name, 'train_time_sec']})
tuning_cost = pd.DataFrame(rows)
U.log_df(log, tuning_cost, 'TABLE 3: ESTIMATED ENERGY OF THE TUNING SEARCH (Phase 4)')
U.save_table(tuning_cost, 'phase5_tuning_energy_estimate', 'Estimated energy of hyperparameter tuning', float_fmt='{:.3e}')

df.to_csv(C.TABLES_DIR / 'phase5_energy_all_raw.csv', index=False)
log.info('PHASE 5 COMPLETE')
tuning_cost

--- TABLE 3: ESTIMATED ENERGY OF THE TUNING SEARCH (Phase 4) ---
               Model  Tuning time (min)  Est. tuning energy (kWh)  Est. tuning CO2 (g)  Tuning / one training (x)
0  Linear Regression           0.000000                  0.000000             0.000000                   0.000000
1      Decision Tree           1.944217                  0.000457             0.168766                  57.127468
2      Random Forest          37.025851                  0.010785             3.984931                  11.403208
3            XGBoost           2.790405                  0.000636             0.234802                  28.280744
4           LightGBM           2.387443                  0.000627             0.231772                  18.318723
5               LSTM           9.248740                  0.002652             0.979663                   3.492115
PHASE 5 COMPLETE


,Model,Tuning time (min),Est. tuning energy (kWh),Est. tuning CO2 (g),Tuning / one training (x)
0,Linear Regression,0.000000,0.000000,0.000000,0.000000
1,Decision Tree,1.944217,0.000457,0.168766,57.127468
2,Random Forest,37.025851,0.010785,3.984931,11.403208
3,XGBoost,2.790405,0.000636,0.234802,28.280744
4,LightGBM,2.387443,0.000627,0.231772,18.318723
5,LSTM,9.248740,0.002652,0.979663,3.492115


## Done: what to check

- **Table 1**: every model should have a positive energy and CO₂ value. Random Forest and the LSTM should be far above the others.
- **Avg power (W)** should be roughly similar for all models, because CodeCarbon estimates power from the machine, not the model. That's exactly the caveat explained at the top, and it means the differences between models come mainly from how *long* each one trains.
- The **measurement context** printout (country/region, carbon intensity, CPU model) goes in the paper's experimental setup section.

Next: `06_stability_testing.ipynb` runs 5 random seeds.